# ML-Based Short-Term Futures Price Prediction Using Order Book Data

This notebook implements a machine learning pipeline to predict short-term futures prices using real-time order book data fetched via WebSocket connections.

## 1. Import Required Libraries

Import libraries for data fetching, processing, and modeling.

In [19]:
import websocket
import json
import pandas as pd
import numpy as np
from datetime import datetime
import time
import threading
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Conv2D, Flatten
import matplotlib.pyplot as plt

### GPU Availability Check

In [20]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Establish WebSocket Connection for Order Book Data

Set up a WebSocket connection to Binance Futures API to stream real-time order book data.

In [21]:
# WebSocket URL for Binance Futures Order Book
SYMBOL = 'btcusdt'
DEPTH_LEVELS = 20
UPDATE_SPEED = 100  # ms

ws_url = f'wss://fstream.binance.com/ws/{SYMBOL}@depth{DEPTH_LEVELS}@{UPDATE_SPEED}ms'

# Global variables to store data
order_book_data = []
data_lock = threading.Lock()

def on_message(ws, message):
    data = json.loads(message)
    with data_lock:
        order_book_data.append({
            'timestamp': datetime.now(),
            'bids': data['b'],
            'asks': data['a'],
            'lastUpdateId': data['u']
        })

def on_error(ws, error):
    print(f"Error: {error}")

def on_close(ws, close_status_code, close_msg):
    print("Connection closed")

def on_open(ws):
    print("Connection opened")

# Function to start WebSocket in a thread
def start_websocket():
    ws = websocket.WebSocketApp(ws_url,
                                on_message=on_message,
                                on_error=on_error,
                                on_close=on_close,
                                on_open=on_open)
    ws.run_forever()

# Start WebSocket in background thread
ws_thread = threading.Thread(target=start_websocket)
ws_thread.daemon = True
ws_thread.start()

Connection opened


## 3. Fetch and Parse Order Book Snapshots

Collect and parse incoming WebSocket messages into structured data.

### Fetch Historical Order Book Data (Optional for More Data)

Use CCXT to fetch order book snapshots over a period for backtesting.

In [22]:
import ccxt

# Fetch historical order book data using CCXT
exchange = ccxt.binance({
    'options': {'defaultType': 'future'},
})

historical_data = []
num_snapshots = 100  # Number of snapshots to fetch

for i in range(num_snapshots):
    try:
        order_book = exchange.fetch_order_book('BTC/USDT', limit=20)
        historical_data.append({
            'timestamp': datetime.now(),
            'bids': order_book['bids'],
            'asks': order_book['asks']
        })
        time.sleep(1)  # Wait 1 second between fetches to avoid rate limits
    except Exception as e:
        print(f"Error fetching data: {e}")
        break

# Convert to DataFrame
df_hist = pd.DataFrame(historical_data)
if not df_hist.empty:
    df_hist['bid_prices'], df_hist['bid_quantities'] = zip(*df_hist['bids'].apply(lambda x: ([p for p, q in x], [q for p, q in x])))
    df_hist['ask_prices'], df_hist['ask_quantities'] = zip(*df_hist['asks'].apply(lambda x: ([p for p, q in x], [q for p, q in x])))
    print(f"Fetched {len(df_hist)} historical snapshots")
else:
    print("No historical data fetched")

# Optionally, combine with real-time data
# df = pd.concat([df, df_hist], ignore_index=True)

Fetched 100 historical snapshots


In [23]:
# Collect data for a longer period to get more samples
time.sleep(60)  # Wait for 1 minute of data

with data_lock:
    df = pd.DataFrame(order_book_data)

# Parse bids and asks into separate columns
def parse_levels(levels):
    prices = [float(level[0]) for level in levels]
    quantities = [float(level[1]) for level in levels]
    return prices, quantities

df['bid_prices'], df['bid_quantities'] = zip(*df['bids'].apply(parse_levels))
df['ask_prices'], df['ask_quantities'] = zip(*df['asks'].apply(parse_levels))

print(f"Collected {len(df)} order book snapshots")
print(df.head())

Collected 2954 order book snapshots
                   timestamp  \
0 2025-11-14 17:37:57.538569   
1 2025-11-14 17:37:57.653141   
2 2025-11-14 17:37:57.756758   
3 2025-11-14 17:37:57.892523   
4 2025-11-14 17:37:58.063797   

                                                bids  \
0  [[96813.80, 6.126], [96813.70, 0.089], [96813....   
1  [[96813.80, 6.273], [96813.70, 0.089], [96813....   
2  [[96813.80, 6.114], [96813.70, 0.089], [96813....   
3  [[96816.80, 5.114], [96816.40, 1.577], [96816....   
4  [[96816.80, 5.314], [96816.70, 0.004], [96816....   

                                                asks   lastUpdateId  \
0  [[96813.90, 0.181], [96814.00, 0.002], [96814....  9191209524079   
1  [[96813.90, 0.181], [96814.00, 0.002], [96814....  9191209535651   
2  [[96813.90, 0.138], [96814.00, 0.004], [96814....  9191209546550   
3  [[96816.90, 0.003], [96817.10, 0.002], [96817....  9191209557091   
4  [[96816.90, 0.006], [96817.00, 0.002], [96817....  9191209574699   

       

## 4. Feature Engineering from Order Book Data

Compute features like bid-ask imbalance, mid-price, spread, and volatility metrics.

In [24]:
print(f"df shape: {df.shape}")
print(df.head() if not df.empty else "df is empty")

df shape: (2954, 8)
                   timestamp  \
0 2025-11-14 17:37:57.538569   
1 2025-11-14 17:37:57.653141   
2 2025-11-14 17:37:57.756758   
3 2025-11-14 17:37:57.892523   
4 2025-11-14 17:37:58.063797   

                                                bids  \
0  [[96813.80, 6.126], [96813.70, 0.089], [96813....   
1  [[96813.80, 6.273], [96813.70, 0.089], [96813....   
2  [[96813.80, 6.114], [96813.70, 0.089], [96813....   
3  [[96816.80, 5.114], [96816.40, 1.577], [96816....   
4  [[96816.80, 5.314], [96816.70, 0.004], [96816....   

                                                asks   lastUpdateId  \
0  [[96813.90, 0.181], [96814.00, 0.002], [96814....  9191209524079   
1  [[96813.90, 0.181], [96814.00, 0.002], [96814....  9191209535651   
2  [[96813.90, 0.138], [96814.00, 0.004], [96814....  9191209546550   
3  [[96816.90, 0.003], [96817.10, 0.002], [96817....  9191209557091   
4  [[96816.90, 0.006], [96817.00, 0.002], [96817....  9191209574699   

                       

In [25]:
# Feature Engineering
def calculate_imbalance(bid_quantities, ask_quantities, k=10):
    bid_sum = sum(bid_quantities[:k])
    ask_sum = sum(ask_quantities[:k])
    return (bid_sum - ask_sum) / (bid_sum + ask_sum) if (bid_sum + ask_sum) > 0 else 0

def calculate_mid_price(bid_prices, ask_prices):
    return (bid_prices[0] + ask_prices[0]) / 2

def calculate_spread(bid_prices, ask_prices):
    return ask_prices[0] - bid_prices[0]

df['imbalance'] = df.apply(lambda row: calculate_imbalance(row['bid_quantities'], row['ask_quantities']), axis=1)
df['mid_price'] = df.apply(lambda row: calculate_mid_price(row['bid_prices'], row['ask_prices']), axis=1)
df['spread'] = df.apply(lambda row: calculate_spread(row['bid_prices'], row['ask_prices']), axis=1)

# Rolling volatility of mid_price
df['mid_price_volatility'] = df['mid_price'].rolling(window=10).std()

# Price change (target for next period)
df['price_change'] = df['mid_price'].shift(-1) - df['mid_price']

print(df[['timestamp', 'imbalance', 'mid_price', 'spread', 'mid_price_volatility', 'price_change']].head())

                   timestamp  imbalance  mid_price  spread  \
0 2025-11-14 17:37:57.538569   0.925764   96813.85     0.1   
1 2025-11-14 17:37:57.653141   0.927869   96813.85     0.1   
2 2025-11-14 17:37:57.756758   0.937784   96813.85     0.1   
3 2025-11-14 17:37:57.892523   0.995636   96816.85     0.1   
4 2025-11-14 17:37:58.063797   0.988934   96816.85     0.1   

   mid_price_volatility  price_change  
0                   NaN           0.0  
1                   NaN           0.0  
2                   NaN           3.0  
3                   NaN           0.0  
4                   NaN           0.0  


## 5. Prepare Dataset for Modeling

Clean the data, create binary labels for price movements, and split into training/validation sets.

In [26]:
# Prepare Dataset
df = df.dropna()  # Drop rows with NaN

# Features
features = ['imbalance', 'spread', 'mid_price_volatility']
X = df[features]

# Target: Binary classification (1 if price increases, 0 otherwise)
y = (df['price_change'] > 0).astype(int)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}, Test set size: {len(X_test)}")

Training set size: 2355, Test set size: 589


In [27]:
# Prepare Dataset
df = df.dropna()  # Drop rows with NaN

# Features
features = ['imbalance', 'spread', 'mid_price_volatility']
X = df[features]

# Target: Binary classification (1 if price increases, 0 otherwise)
y = (df['price_change'] > 0).astype(int)

# Check class distribution
print("Overall target distribution:")
print(y.value_counts())
print(f"Percentage of 1s: {y.mean():.2%}")

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}, Test set size: {len(X_test)}")
print(f"Train target distribution: {y_train.value_counts()}")
print(f"Test target distribution: {y_test.value_counts()}")

Overall target distribution:
price_change
0    2790
1     154
Name: count, dtype: int64
Percentage of 1s: 5.23%
Training set size: 2355, Test set size: 589
Train target distribution: price_change
0    2232
1     123
Name: count, dtype: int64
Test target distribution: price_change
0    558
1     31
Name: count, dtype: int64


## 6. Train Supervised Learning Model

Train XGBoost or Random Forest on the engineered features.

In [28]:
# Train XGBoost
model = xgb.XGBClassifier(objective='binary:logistic', n_estimators=100)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

Model Accuracy: 0.93


In [29]:
# Train XGBoost with class weights to handle imbalance
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

model = xgb.XGBClassifier(objective='binary:logistic', n_estimators=100, scale_pos_weight=class_weight_dict[1]/class_weight_dict[0])
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

# Alternative: Use sample weights
# sample_weights = np.where(y_train == 1, class_weight_dict[1], class_weight_dict[0])
# model.fit(X_train, y_train, sample_weight=sample_weights)

Model Accuracy: 0.85


## 7. Implement Deep Learning Model

Build and train an LSTM model to capture temporal patterns.

### CNN Model for Order Book as Image

Treat the order book as a 2D matrix (bids and asks) and use CNN to extract spatial features.

In [30]:
# Create 2D representation of order book
def create_order_book_image(bid_quantities, ask_quantities, max_levels=20):
    # Pad or truncate to max_levels
    bids = np.array(bid_quantities[:max_levels])
    asks = np.array(ask_quantities[:max_levels])
    if len(bids) < max_levels:
        bids = np.pad(bids, (0, max_levels - len(bids)), 'constant')
    if len(asks) < max_levels:
        asks = np.pad(asks, (0, max_levels - len(asks)), 'constant')
    # Stack as 2 channels: bids and asks
    image = np.stack([bids, asks], axis=-1)  # Shape: (max_levels, 2)
    return image

# Apply to dataframe
df['order_book_image'] = df.apply(lambda row: create_order_book_image(row['bid_quantities'], row['ask_quantities']), axis=1)

# Prepare data for CNN
X_images = np.stack(df['order_book_image'].values)
X_images = X_images[..., np.newaxis]  # Add channel dimension: (n, 20, 2, 1)
y_images = y.values[:len(X_images)]  # Align with images

X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(X_images, y_images, test_size=0.2, random_state=42)

# Build CNN model
model_cnn = Sequential()
model_cnn.add(Conv2D(32, (3, 1), activation='relu', input_shape=(20, 2, 1)))
model_cnn.add(Flatten())
model_cnn.add(Dense(64, activation='relu'))
model_cnn.add(Dense(1, activation='sigmoid'))

model_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_cnn.fit(X_train_img, y_train_img, epochs=10, batch_size=32, validation_data=(X_test_img, y_test_img))

Epoch 1/10


/home/misango/anaconda3/envs/research_env/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


74/74 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9355 - loss: 0.2675 - val_accuracy: 0.9440 - val_loss: 0.1814
Epoch 2/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9355 - loss: 0.2675 - val_accuracy: 0.9440 - val_loss: 0.1814
Epoch 2/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9456 - loss: 0.1686 - val_accuracy: 0.9440 - val_loss: 0.1735
Epoch 3/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9456 - loss: 0.1686 - val_accuracy: 0.9440 - val_loss: 0.1735
Epoch 3/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9473 - loss: 0.1640 - val_accuracy: 0.9440 - val_loss: 0.1762
Epoch 4/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9473 - loss: 0.1640 - val_accuracy: 0.9440 - val_loss: 0.1762
Epoch 4/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9478 - loss: 0.1619 - val_accuracy: 0.9440 - val_loss: 0.1789
Epoch 5/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9478 - loss: 0.1619 - val_accuracy: 0.9440 - val_loss: 0.1789
E

In [31]:
# Create 2D representation of order book
def create_order_book_image(bid_quantities, ask_quantities, max_levels=20):
    # Pad or truncate to max_levels
    bids = np.array(bid_quantities[:max_levels])
    asks = np.array(ask_quantities[:max_levels])
    if len(bids) < max_levels:
        bids = np.pad(bids, (0, max_levels - len(bids)), 'constant')
    if len(asks) < max_levels:
        asks = np.pad(asks, (0, max_levels - len(asks)), 'constant')
    # Stack as 2 channels: bids and asks
    image = np.stack([bids, asks], axis=-1)  # Shape: (max_levels, 2)
    return image

# Apply to dataframe
df['order_book_image'] = df.apply(lambda row: create_order_book_image(row['bid_quantities'], row['ask_quantities']), axis=1)

# Prepare data for CNN
X_images = np.stack(df['order_book_image'].values)
X_images = X_images[..., np.newaxis]  # Add channel dimension: (n, 20, 2, 1)
y_images = y.values[:len(X_images)]  # Align with images

X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(X_images, y_images, test_size=0.2, random_state=42)

# Class weights for CNN
class_weights_cnn = compute_class_weight('balanced', classes=np.unique(y_train_img), y=y_train_img)
class_weight_dict_cnn = {0: class_weights_cnn[0], 1: class_weights_cnn[1]}

# Build CNN model
model_cnn = Sequential()
model_cnn.add(Conv2D(32, (3, 1), activation='relu', input_shape=(20, 2, 1)))
model_cnn.add(Flatten())
model_cnn.add(Dense(64, activation='relu'))
model_cnn.add(Dense(1, activation='sigmoid'))

model_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_cnn.fit(X_train_img, y_train_img, epochs=10, batch_size=32, validation_data=(X_test_img, y_test_img), class_weight=class_weight_dict_cnn)

Epoch 1/10


/home/misango/anaconda3/envs/research_env/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


74/74 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.6730 - loss: 0.5493 - val_accuracy: 0.6537 - val_loss: 0.6180
Epoch 2/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.6730 - loss: 0.5493 - val_accuracy: 0.6537 - val_loss: 0.6180
Epoch 2/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7265 - loss: 0.4899 - val_accuracy: 0.7368 - val_loss: 0.5325
Epoch 3/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7265 - loss: 0.4899 - val_accuracy: 0.7368 - val_loss: 0.5325
Epoch 3/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7439 - loss: 0.4790 - val_accuracy: 0.7317 - val_loss: 0.5327
Epoch 4/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7439 - loss: 0.4790 - val_accuracy: 0.7317 - val_loss: 0.5327
Epoch 4/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7261 - loss: 0.4738 - val_accuracy: 0.7589 - val_loss: 0.5047
Epoch 5/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7261 - loss: 0.4738 - val_accuracy: 0.7589 - val_loss: 0.5047
E

In [32]:
# Prepare data for LSTM (sequences)
sequence_length = 10
X_seq = []
y_seq = []
for i in range(len(X) - sequence_length):
    X_seq.append(X.iloc[i:i+sequence_length].values)
    y_seq.append(y.iloc[i+sequence_length])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

# Build LSTM model
model_lstm = Sequential()
model_lstm.add(LSTM(50, input_shape=(sequence_length, len(features))))
model_lstm.add(Dense(1, activation='sigmoid'))

model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.fit(X_train_seq, y_train_seq, epochs=10, batch_size=32, validation_data=(X_test_seq, y_test_seq))

Epoch 1/10


/home/misango/anaconda3/envs/research_env/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9267 - loss: 0.3326 - val_accuracy: 0.9608 - val_loss: 0.1600
Epoch 2/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9267 - loss: 0.3326 - val_accuracy: 0.9608 - val_loss: 0.1600
Epoch 2/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9446 - loss: 0.1985 - val_accuracy: 0.9608 - val_loss: 0.1538
Epoch 3/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9446 - loss: 0.1985 - val_accuracy: 0.9608 - val_loss: 0.1538
Epoch 3/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9446 - loss: 0.1902 - val_accuracy: 0.9608 - val_loss: 0.1461
Epoch 4/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9446 - loss: 0.1902 - val_accuracy: 0.9608 - val_loss: 0.1461
Epoch 4/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9446 - loss: 0.1881 - val_accuracy: 0.9608 - val_loss: 0.1460
Epoch 5/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9446 - loss: 0.1881 - val_accuracy: 0.9608 - val_loss: 0.1460
Epo

In [33]:
# Prepare data for LSTM (sequences)
sequence_length = 10
X_seq = []
y_seq = []
for i in range(len(X) - sequence_length):
    X_seq.append(X.iloc[i:i+sequence_length].values)
    y_seq.append(y.iloc[i+sequence_length])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

# Class weights for LSTM
class_weights_lstm = compute_class_weight('balanced', classes=np.unique(y_train_seq), y=y_train_seq)
class_weight_dict_lstm = {0: class_weights_lstm[0], 1: class_weights_lstm[1]}

# Build LSTM model
model_lstm = Sequential()
model_lstm.add(LSTM(50, input_shape=(sequence_length, len(features))))
model_lstm.add(Dense(1, activation='sigmoid'))

model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.fit(X_train_seq, y_train_seq, epochs=10, batch_size=32, validation_data=(X_test_seq, y_test_seq), class_weight=class_weight_dict_lstm)

Epoch 1/10


/home/misango/anaconda3/envs/research_env/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


74/74 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6357 - loss: 0.6324 - val_accuracy: 0.5520 - val_loss: 0.6856
Epoch 2/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6357 - loss: 0.6324 - val_accuracy: 0.5520 - val_loss: 0.6856
Epoch 2/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5769 - loss: 0.5647 - val_accuracy: 0.6303 - val_loss: 0.5446
Epoch 3/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5769 - loss: 0.5647 - val_accuracy: 0.6303 - val_loss: 0.5446
Epoch 3/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6093 - loss: 0.5554 - val_accuracy: 0.6150 - val_loss: 0.5800
Epoch 4/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6093 - loss: 0.5554 - val_accuracy: 0.6150 - val_loss: 0.5800
Epoch 4/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5871 - loss: 0.5416 - val_accuracy: 0.6491 - val_loss: 0.5128
Epoch 5/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5871 - loss: 0.5416 - val_accuracy: 0.6491 - val_loss: 0.5128
E

## 8. Generate Price Predictions

Use the trained models to predict on new data.

In [34]:
# Predictions with XGBoost
predictions_xgb = model.predict(X_test)

# Predictions with LSTM
predictions_lstm = (model_lstm.predict(X_test_seq) > 0.5).astype(int).flatten()

# Predictions with CNN
predictions_cnn = (model_cnn.predict(X_test_img) > 0.5).astype(int).flatten()

print("XGBoost Predictions:", predictions_xgb[:10])
print("LSTM Predictions:", predictions_lstm[:10])
print("CNN Predictions:", predictions_cnn[:10])

19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
XGBoost Predictions: [0 0 0 0 0 1 0 0 0 0]
LSTM Predictions: [1 0 1 1 0 0 0 0 1 0]
CNN Predictions: [0 0 1 1 0 1 1 0 0 1]
XGBoost Predictions: [0 0 0 0 0 1 0 0 0 0]
LSTM Predictions: [1 0 1 1 0 0 0 0 1 0]
CNN Predictions: [0 0 1 1 0 1 1 0 0 1]


In [35]:
# Predictions with XGBoost
predictions_xgb = model.predict(X_test)

# Predictions with LSTM
predictions_lstm = (model_lstm.predict(X_test_seq) > 0.5).astype(int).flatten()

# Predictions with CNN
predictions_cnn = (model_cnn.predict(X_test_img) > 0.5).astype(int).flatten()

print("XGBoost Predictions:", predictions_xgb[:10])
print("LSTM Predictions:", predictions_lstm[:10])
print("CNN Predictions:", predictions_cnn[:10])

# Diagnostic: Check class distribution
print("\nTarget distribution in test set:")
print(y_test.value_counts())

# Diagnostic: Accuracy on test set
from sklearn.metrics import classification_report
print("\nXGBoost Classification Report:")
print(classification_report(y_test, predictions_xgb))

# For LSTM and CNN, since they use different splits, check separately if needed
# But for simplicity, focus on XGBoost

19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
XGBoost Predictions: [0 0 0 0 0 1 0 0 0 0]
LSTM Predictions: [1 0 1 1 0 0 0 0 1 0]
CNN Predictions: [0 0 1 1 0 1 1 0 0 1]

Target distribution in test set:
price_change
0    558
1     31
Name: count, dtype: int64

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.88      0.92       558
           1       0.11      0.26      0.15        31

    accuracy                           0.85       589
   macro avg       0.53      0.57      0.53       589
weighted avg       0.91      0.85      0.88       589

XGBoost Predictions: [0 0 0 0 0 1 0 0 0 0]
LSTM Predictions: [1 0 1 1 0 0 0 0 1 0]
CNN Predictions: [0 0 1 1 0 1 1 0 0 1]

Target distribution in test set:
price_change
0    558
1     31
Name: count, dtype: int64

XGBoost Classification Report:
              precision

In [ ]:
# Predictions with XGBoost
predictions_xgb = model.predict(X_test)

# Predictions with LSTM
predictions_lstm = (model_lstm.predict(X_test_seq) > 0.5).astype(int).flatten()

# Predictions with CNN
predictions_cnn = (model_cnn.predict(X_test_img) > 0.5).astype(int).flatten()

print("XGBoost Predictions:", predictions_xgb[:10])
print("LSTM Predictions:", predictions_lstm[:10])
print("CNN Predictions:", predictions_cnn[:10])

# Diagnostic: Check class distribution
print("\nTarget distribution in test set:")
print(y_test.value_counts())

# Diagnostic: Accuracy on test set
from sklearn.metrics import classification_report
print("\nXGBoost Classification Report:")
print(classification_report(y_test, predictions_xgb))

# Check prediction distribution
print("\nPrediction distribution for XGBoost:")
print(pd.Series(predictions_xgb).value_counts())

# For LSTM and CNN, since they use different splits, check separately if needed
# But for simplicity, focus on XGBoost

## 9. Simulate Trading Execution

Implement a simple simulation of trade execution based on predictions and calculate potential profits.

In [36]:
# Simple Trading Simulation
# Assume we trade based on XGBoost predictions
capital = 10000
position = 0  # 1 for long, -1 for short
entry_price = 0

profits = []

for i in range(len(predictions_xgb)):
    pred = predictions_xgb[i]
    actual_change = df.iloc[len(X_train) + i]['price_change']
    current_price = df.iloc[len(X_train) + i]['mid_price']
    
    if position == 0:
        if pred == 1:  # Predict up, go long
            position = 1
            entry_price = current_price
        elif pred == 0:  # Predict down, go short
            position = -1
            entry_price = current_price
    else:
        # Close position
        if position == 1:
            profit = (current_price - entry_price) * 100  # Assume 1 lot
        else:
            profit = (entry_price - current_price) * 100
        profits.append(profit)
        capital += profit
        position = 0

total_profit = sum(profits)
print(f"Total Simulated Profit: {total_profit:.2f}, Final Capital: {capital:.2f}")

Total Simulated Profit: -3830.00, Final Capital: 6170.00


In [ ]:
# Simple Trading Simulation
# Assume we trade based on XGBoost predictions
capital = 10000
position = 0  # 1 for long, -1 for short
entry_price = 0

profits = []
trades = []

for i in range(len(predictions_xgb)):
    pred = predictions_xgb[i]
    actual_change = df.iloc[len(X_train) + i]['price_change']
    current_price = df.iloc[len(X_train) + i]['mid_price']
    
    if position == 0:
        if pred == 1:  # Predict up, go long
            position = 1
            entry_price = current_price
            trades.append(('long', entry_price))
        elif pred == 0:  # Predict down, go short
            position = -1
            entry_price = current_price
            trades.append(('short', entry_price))
    else:
        # Close position on next signal or end
        if position == 1:
            profit = (current_price - entry_price) * 100  # Assume 1 lot
        else:
            profit = (entry_price - current_price) * 100
        profits.append(profit)
        capital += profit
        position = 0  # Close position

# If still in position at end, close it
if position != 0:
    last_price = df.iloc[len(X_train) + len(predictions_xgb) - 1]['mid_price']
    if position == 1:
        profit = (last_price - entry_price) * 100
    else:
        profit = (entry_price - last_price) * 100
    profits.append(profit)
    capital += profit

total_profit = sum(profits)
print(f"Total Simulated Profit: {total_profit:.2f}, Final Capital: {capital:.2f}")
print(f"Number of trades: {len(trades)}")
print(f"Average profit per trade: {total_profit / len(trades) if trades else 0:.2f}")

# Compare to buy-and-hold
initial_price = df.iloc[len(X_train)]['mid_price']
final_price = df.iloc[len(X_train) + len(predictions_xgb) - 1]['mid_price']
buy_hold_profit = (final_price - initial_price) * 100
print(f"Buy-and-hold profit: {buy_hold_profit:.2f}")
print(f"Strategy vs Buy-and-hold: {'Better' if total_profit > buy_hold_profit else 'Worse'}")